# Cherry-Picked Transcript Statistics

Feature-level inspection of four representative transcripts (low/high entropy × coding/lncRNA) across the TE, non-B DNA, and uncertainty-analysis pipelines.

In [ ]:
cherry_pick: dict[str, list[str]] = {
    "low_entropy_coding":  ["ENST00000265171"],
    "low_entropy_lncrna":  ["ENST00000710946"],
    "high_entropy_coding": ["ENST00000652598"],
    "high_entropy_lncrna": ["ENST00000648123"],
}

cherry_names: dict[str, list[str]] = {
    "low_entropy_coding":  ["EGF"],
    "low_entropy_lncrna":  ["MALAT1"],
    "high_entropy_coding": ["SWI5"],
    "high_entropy_lncrna": ["MEG3"],
}



## 1. Imports and Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE = Path("/mnt/cbib/LNClassifier/paper")

TE_FEATURES  = BASE / "te_pipeline/results/te_analysis_flexible/features/all_transcripts_te_features.corrected.csv"
NBD_FEATURES = BASE / "nonb-pipeline/results/gencode.v47.transcripts/extended_analysis/features_nonb_features.csv"
ENTROPY_TSV  = BASE / "results/gencode.v47.common.cdhit.cv/features/entropy/gencode.v47.common.cdhit.cv_uncertainty_analysis.tsv"

print("Paths:")
for label, p in [("TE features", TE_FEATURES), ("NBD features", NBD_FEATURES), ("Entropy TSV", ENTROPY_TSV)]:
    print(f"  {label:12s}: {'✓' if p.exists() else '✗'} {p}")

Paths:
  TE features : ✓ /mnt/cbib/LNClassifier/paper/te_pipeline/results/te_analysis_flexible/features/all_transcripts_te_features.corrected.csv
  NBD features: ✓ /mnt/cbib/LNClassifier/paper/nonb-pipeline/results/gencode.v47.transcripts/extended_analysis/features_nonb_features.csv
  Entropy TSV : ✓ /mnt/cbib/LNClassifier/paper/results/gencode.v47.common.cdhit.cv/features/entropy/gencode.v47.common.cdhit.cv_uncertainty_analysis.tsv


## 2. Load Feature Tables

In [ ]:
# ── TE pipeline ──────────────────────────────────────────────────────────────
te = pd.read_csv(TE_FEATURES, index_col="transcript_id", low_memory=False)
# Strip version suffix (e.g. ENST00000265171.10  →  ENST00000265171)
te.index = te.index.str.replace(r"\.\d+$", "", regex=True)
te["coding_class"] = te["coding_class"].str.strip()

print(f"TE feature table : {te.shape[0]:,} transcripts × {te.shape[1]} features")
print(f"  coding : {(te['coding_class'] == 'coding').sum():,}")
print(f"  lncRNA : {(te['coding_class'] == 'lncRNA').sum():,}")
te[["transcript_length", "coding_class", "te_count", "te_count_per_kb",
    "te_sum_hit_length_pct", "lctr_count"]].head(3)

TE feature table : 385,659 transcripts × 173 features
  coding : 112,218
  lncRNA : 191,106


,transcript_length,coding_class,te_count,te_count_per_kb,te_sum_hit_length_pct,lctr_count
transcript_id,,,,,,
ENST00000832824,1379,lncRNA,0,0.000000,0.000000,1
ENST00000832825,1417,lncRNA,0,0.000000,0.000000,1
ENST00000832826,1538,lncRNA,1,0.650195,11.053316,1


In [ ]:
te.columns.tolist()

['transcript_length',
 'te_min_sw_score',
 'te_max_sw_score',
 'te_mean_sw_score',
 'te_min_divergence',
 'te_max_divergence',
 'te_mean_divergence',
 'te_min_perc_del',
 'te_max_perc_del',
 'te_mean_perc_del',
 'te_min_perc_ins',
 'te_max_perc_ins',
 'te_mean_perc_ins',
 'te_count',
 'te_sum_hit_length',
 'te_min_hit_length',
 'te_mean_hit_length',
 'te_max_hit_length',
 'te_min_hit_reference_coverage',
 'te_mean_hit_reference_coverage',
 'te_max_hit_reference_coverage',
 'te_sum_num_fragments',
 'te_mean_num_fragments',
 'te_max_num_fragments',
 'te_sum_fragmented',
 'te_unique_subfamilies',
 'te_unique_classes',
 'te_unique_families',
 'te_has_dna',
 'te_dna_count',
 'te_has_ervk',
 'te_ervk_count',
 'te_has_ervl',
 'te_ervl_count',
 'te_has_ervl-malr',
 'te_ervl-malr_count',
 'te_has_erv1',
 'te_erv1_count',
 'te_has_line',
 'te_line_count',
 'te_has_ltr',
 'te_ltr_count',
 'te_has_ple',
 'te_ple_count',
 'te_has_rc',
 'te_rc_count',
 'te_has_retroposon',
 'te_retroposon_count',
 '

In [ ]:
# ── Non-B DNA pipeline ───────────────────────────────────────────────────────
nbd = pd.read_csv(NBD_FEATURES, index_col="transcript_id", low_memory=False)
nbd.index = nbd.index.str.replace(r"\.\d+$", "", regex=True)
nbd["coding_class"] = nbd["coding_class"].str.strip()

print(f"NBD feature table: {nbd.shape[0]:,} transcripts × {nbd.shape[1]} features")
print(f"  coding : {(nbd['coding_class'] == 'coding').sum():,}")
print(f"  lncRNA : {(nbd['coding_class'] == 'lncRNA').sum():,}")
nbd[["transcript_length", "coding_class", "total_nonb_count", "total_nonb_coverage_pct"]].head(3)

NBD feature table: 385,659 transcripts × 189 features
  coding : 112,218
  lncRNA : 191,106


,transcript_length,coding_class,total_nonb_count,total_nonb_coverage_pct
transcript_id,,,,
ENST00000000233,3290,coding,19.0,13.434650
ENST00000000412,9285,coding,25.0,5.320409
ENST00000000442,11220,coding,72.0,14.037433


In [ ]:
# ── Uncertainty / entropy analysis ───────────────────────────────────────────
ent = pd.read_csv(ENTROPY_TSV, sep="\t", index_col="seq_ID", low_memory=False)
ent.index = ent.index.str.replace(r"\.\d+$", "", regex=True)

ENTROPY_TOOL_COLS = [c for c in ent.columns if c.endswith("_entropy")]
ENTROPY_KEY_COLS  = ENTROPY_TOOL_COLS + ["H_exp", "H_pred", "I_bald", "coding_class", "biotype"]

print(f"Entropy table    : {ent.shape[0]:,} transcripts × {ent.shape[1]} features")
print(f"  Per-tool entropy columns ({len(ENTROPY_TOOL_COLS)}): {ENTROPY_TOOL_COLS}")
ent[ENTROPY_KEY_COLS].head(3)

Entropy table    : 111,652 transcripts × 13 features
  Per-tool entropy columns (8): ['coding_score_rnasamba_entropy', 'coding_potential_feelnc_entropy', 'Coding_prob_l_cpat_entropy', 'Noncoding_prob_ss_lncDC_entropy', 'coding_prob_mrnn_entropy', 'P(pcRNA)_lncrnabert_entropy', 'prob_coding_plncpro_entropy', 'Coding.Potential_ss_lncfinder_entropy']


,coding_score_rnasamba_entropy,coding_potential_feelnc_entropy,Coding_prob_l_cpat_entropy,Noncoding_prob_ss_lncDC_entropy,coding_prob_mrnn_entropy,P(pcRNA)_lncrnabert_entropy,prob_coding_plncpro_entropy,Coding.Potential_ss_lncfinder_entropy,H_exp,H_pred,I_bald,coding_class,biotype
seq_ID,,,,,,,,,,,,,
ENST00000000412,0.052103,0.251388,0.351220,0.002338,0.148375,0.000145,0.260341,0.084436,0.143793,0.161986,0.018193,1,coding
ENST00000002596,0.098077,0.269156,0.032545,0.036134,0.434400,0.000143,0.979051,0.044755,0.236783,0.372960,0.136177,1,coding
ENST00000002829,0.008346,0.000000,0.013588,0.016586,0.104798,0.000133,0.020814,0.191563,0.044478,0.053527,0.009048,1,coding


## 3. Cherry-Picked Transcript Summary

In [ ]:
# Build a flat lookup: transcript_id -> (category, gene_name)
cherry_lookup: dict[str, tuple[str, str]] = {
    ids[0]: (cat, cherry_names[cat][0])
    for cat, ids in cherry_pick.items()
}

TE_DISPLAY_COLS: dict[str, str] = {
    "transcript_length":        "length (nt)",
    "global_rm_count":          "global RepeatMasker hits",
    "te_count":                 "TE hits",
    "te_count_per_kb":          "TE hits/kb",
    "te_sum_hit_length_pct":    "TE coverage (%)",
    "te_mean_divergence":       "TE mean divergence",
    "te_young_count":           "young TE hits",
    "te_ancient_count":         "ancient TE hits",
    "te_line_count":            "LINE hits",
    "te_sine_count":            "SINE hits",
    "te_ltr_count":             "LTR hits",
    "te_dna_count":             "DNA TE hits",
    "lctr_count":               "LCTR hits",
    "lctr_total_length_pct":    "LCTR coverage (%)",
    "lctr_low_complexity_count":"low-complexity hits",
    "lctr_simple_repeat_count": "simple-repeat hits",
    "lctr_satellite_count":     "satellite hits",
}

NBD_DISPLAY_COLS: dict[str, str] = {
    "total_nonb_count":         "NBD total hits",
    "total_nonb_coverage_pct":  "NBD coverage (%)",
    "apr_hit_count":            "APR hits",
    "dr_hit_count":             "DR hits",
    "gq_hit_count":             "G4 hits",
    "ir_hit_count":             "IR hits",
    "mr_hit_count":             "MR hits",
    "str_hit_count":            "STR hits",
    "tri_hit_count":            "TRI hits",
    "z_hit_count":              "Z-DNA hits",
}

ENT_DISPLAY_COLS: dict[str, str] = {
    c: c.replace("_entropy", "").replace("_", " ") for c in ENTROPY_TOOL_COLS
} | {"H_exp": "H_exp", "H_pred": "H_pred", "I_bald": "I_bald"}

rows = []
for enst_id, (category, gene_name) in cherry_lookup.items():
    row: dict = {"transcript_id": enst_id, "gene": gene_name, "category": category}

    # --- TE pipeline ---
    if enst_id in te.index:
        t = te.loc[enst_id]
        for col, label in TE_DISPLAY_COLS.items():
            row[f"TE | {label}"] = t.get(col, np.nan)
    else:
        print(f"  WARNING: {enst_id} not found in TE table")

    # --- NBD pipeline ---
    if enst_id in nbd.index:
        n = nbd.loc[enst_id]
        for col, label in NBD_DISPLAY_COLS.items():
            row[f"NBD | {label}"] = n.get(col, np.nan)
    else:
        print(f"  WARNING: {enst_id} not found in NBD table")

    # --- Entropy ---
    if enst_id in ent.index:
        e = ent.loc[enst_id]
        for col, label in ENT_DISPLAY_COLS.items():
            row[f"Entropy | {label}"] = e.get(col, np.nan)
    else:
        print(f"  WARNING: {enst_id} not found in entropy table")

    rows.append(row)

cherry_df = pd.DataFrame(rows).set_index("transcript_id")
print(f"Cherry-pick table: {cherry_df.shape[0]} transcripts × {cherry_df.shape[1]} features\n")
display(cherry_df[["gene", "category"]].T)


Cherry-pick table: 4 transcripts × 40 features



transcript_id,ENST00000265171,ENST00000710946,ENST00000652598,ENST00000648123
gene,EGF,MALAT1,SWI5,MEG3
category,low_entropy_coding,low_entropy_lncrna,high_entropy_coding,high_entropy_lncrna


In [ ]:
# ── TE pipeline features ─────────────────────────────────────────────────────
te_cols = [c for c in cherry_df.columns if c.startswith("TE | ")]
print("TE pipeline features:")
display(cherry_df[te_cols].T.rename(
    columns=lambda tid: f"{cherry_lookup[tid][1]} ({tid})"
).round(3))

TE pipeline features:


transcript_id,EGF (ENST00000265171),MALAT1 (ENST00000710946),SWI5 (ENST00000652598),MEG3 (ENST00000648123)
TE | length (nt),6388.000,2812.000,1373.000,644.0
TE | global RepeatMasker hits,1.000,1.000,3.000,0.0
TE | TE hits,1.000,0.000,3.000,0.0
TE | TE hits/kb,0.157,0.000,2.185,0.0
TE | TE coverage (%),5.463,0.000,17.698,0.0
TE | TE mean divergence,18.100,0.000,17.167,0.0
TE | young TE hits,0.000,0.000,0.000,0.0
TE | ancient TE hits,0.000,0.000,1.000,0.0
TE | LINE hits,0.000,0.000,0.000,0.0
TE | SINE hits,1.000,0.000,2.000,0.0


In [ ]:
# ── Non-B DNA features ───────────────────────────────────────────────────────
nbd_cols = [c for c in cherry_df.columns if c.startswith("NBD | ")]
print("Non-B DNA pipeline features:")
display(cherry_df[nbd_cols].T.rename(
    columns=lambda tid: f"{cherry_lookup[tid][1]} ({tid})"
).round(3))

Non-B DNA pipeline features:


transcript_id,EGF (ENST00000265171),MALAT1 (ENST00000710870),SWI5 (ENST00000608796),MEG3 (ENST00000524131)
NBD | NBD total hits,544.000,25.00,102.000,135.000
NBD | NBD coverage (%),210.301,48.46,288.631,389.598
NBD | APR hits,24.000,0.00,0.000,4.000
NBD | DR hits,43.000,0.00,11.000,5.000
NBD | G4 hits,13.000,5.00,12.000,20.000
NBD | IR hits,268.000,12.00,29.000,63.000
NBD | MR hits,66.000,4.00,14.000,11.000
NBD | STR hits,104.000,2.00,23.000,24.000
NBD | TRI hits,14.000,2.00,9.000,2.000
NBD | Z-DNA hits,12.000,0.00,4.000,6.000


In [ ]:
# ── Entropy / uncertainty ────────────────────────────────────────────────────
ent_cols = [c for c in cherry_df.columns if c.startswith("Entropy | ")]
print("Entropy / uncertainty (gencode.v47.common.cdhit.cv):")
display(cherry_df[ent_cols].T.rename(
    columns=lambda tid: f"{cherry_lookup[tid][1]} ({tid})"
).round(4))

Entropy / uncertainty (gencode.v47.common.cdhit.cv):


transcript_id,EGF (ENST00000265171),MALAT1 (ENST00000710870),SWI5 (ENST00000608796),MEG3 (ENST00000524131)
Entropy | coding score rnasamba,0.0363,0.0548,0.8437,0.8256
Entropy | coding potential feelnc,0.0672,0.2141,0.0672,0.3733
Entropy | Coding prob l cpat,0.0511,0.3246,0.7388,0.2874
Entropy | Noncoding prob ss lncDC,0.0010,0.0025,0.9961,0.0529
Entropy | coding prob mrnn,0.1498,0.2143,0.0678,0.8329
Entropy | P(pcRNA) lncrnabert,0.0001,0.0001,0.0003,0.0002
Entropy | prob coding plncpro,0.4969,0.2043,0.9850,0.9996
Entropy | Coding.Potential ss lncfinder,0.0355,0.3327,0.2905,0.9294
Entropy | H_exp,0.1047,0.1684,0.4987,0.5377
Entropy | H_pred,0.1357,0.1863,0.9601,0.9709
